# C5 · M1b (clean-neg pull-push) — Colab trainer

Trains **M1b** = the from-scratch pull-push arm with **neg-source = CLEAN** (transitive-separation variant).
Config is **identical to M1a except neg-source**: base = TRUE MSD (`msd_v0`, steps=10), pull-push ON, anchor A (clean, stop-grad),
α=β=0.5, τ=0.1, 80 epochs, RAMP-80 recipe (lr 0.05, drop @ep70, crop+flip), **seed 0** → paired with M1a.

## How to run (manual steps)
1. **Upload 2 files to your Google Drive** (e.g. into `MyDrive/attackdro/`):
   - `cifar-10-python.tar.gz`  ← the EXACT CIFAR-10 tarball from the PC repo `data/cifar-10-python.tar.gz` (sha256 starts `6d958be0…`). Do NOT let Colab re-download — we use your upload so data+splits match M1a.
   - `attackdro_code.zip`  ← zip containing the repo folders **`src/`** and the file **`scripts/dev/c5_fromscratch.py`**. On the PC: `zip -r attackdro_code.zip src scripts/dev/c5_fromscratch.py` (from the repo root).
2. **Runtime → Change runtime type → GPU** (T4 or L4).
3. **Edit the 3 paths** in the `EDIT ME` cell below to match where you put those files + where to save outputs.
4. **Runtime → Run all.** Training auto-resumes if Colab drops (outputs live on Drive).
5. When done, **download the output folder** (`val_best.pt`, `embed_dump.json`, `train.json`, `ckpt/`) and send it back — it will be audited under the identical 12-AA harness on the PC.


In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


## EDIT ME — paths


In [ ]:
# 2) EDIT these 3 paths to match your Drive
DRIVE_CIFAR_TARGZ = '/content/drive/MyDrive/attackdro/cifar-10-python.tar.gz'  # the tarball you uploaded
DRIVE_CODE_ZIP    = '/content/drive/MyDrive/attackdro/attackdro_code.zip'      # zip of src/ + scripts/dev/c5_fromscratch.py
DRIVE_OUT         = '/content/drive/MyDrive/attackdro/C5_M1b_out'              # outputs+resume live here (survives disconnects)

REPO = '/content/attackdro'   # local working copy of the code (fast disk)
import os; os.makedirs(DRIVE_OUT, exist_ok=True)
assert os.path.exists(DRIVE_CIFAR_TARGZ), 'upload cifar-10-python.tar.gz to Drive first'
assert os.path.exists(DRIVE_CODE_ZIP), 'upload attackdro_code.zip to Drive first'
print('paths OK')


In [ ]:
# 3) Deps: torch/torchvision/numpy are preinstalled on Colab; M1b training needs only wandb extra.
#    (autoattack is NOT needed here — the 12-AA eval is run on the PC.)
!pip -q install wandb==0.28.0
# Optional live logging: uncomment + paste your key, or leave disabled (WANDB_MODE below).
# import wandb; wandb.login()


In [ ]:
# 4) Unpack the code zip -> REPO ; set ATTACKDRO_ROOT so c5 finds src/ and data/
import os, zipfile, shutil
os.makedirs(REPO, exist_ok=True)
with zipfile.ZipFile(DRIVE_CODE_ZIP) as z: z.extractall(REPO)
assert os.path.exists(f'{REPO}/src/robustdro'), 'zip must contain src/robustdro'
assert os.path.exists(f'{REPO}/scripts/dev/c5_fromscratch.py'), 'zip must contain scripts/dev/c5_fromscratch.py'
os.environ['ATTACKDRO_ROOT'] = REPO   # c5_fromscratch reads this; falls back to the PC path otherwise
print('code ready at', REPO)


In [ ]:
# 5) Extract CIFAR-10 from YOUR upload into REPO/data (no re-download) + verify it matches the PC data
import hashlib, tarfile, os
h = hashlib.sha256(open(DRIVE_CIFAR_TARGZ,'rb').read()).hexdigest()
print('cifar tar sha256:', h[:16], '(PC =', '6d958be074577803', ')')
assert h.startswith('6d958be074577803'), 'CIFAR tar does not match the PC copy — upload the repo data/cifar-10-python.tar.gz'
os.makedirs(f'{REPO}/data', exist_ok=True)
with tarfile.open(DRIVE_CIFAR_TARGZ) as t: t.extractall(f'{REPO}/data')   # -> REPO/data/cifar-10-batches-py
assert os.path.exists(f'{REPO}/data/cifar-10-batches-py/test_batch'), 'extract failed'
print('CIFAR-10 ready (same bytes as M1a)')


In [ ]:
# 6) Confirm the splits are deterministic & identical to M1a (constants + fixed seed)
import sys; sys.path.insert(0, f'{REPO}/src')
exec(open(f'{REPO}/scripts/dev/c5_fromscratch.py').read().split('def main')[0])  # load constants/helpers only
print('TRAIN_CORE', TRAIN_CORE, '| VAL_SELECT', VAL_SELECT, '(fixed ranges → same as M1a)')
print('seed 0 + identical data bytes → RandomCrop/Flip + loader shuffle are seeded → paired with M1a; only neg-source differs')


## Train M1b (auto-resumes from Drive if the session drops)


In [ ]:
# 7) Train M1b. Outputs (ckpt/, resume.pt, val_best.pt, embed_dump.json, train.json) go to DRIVE_OUT.
#    Re-running this cell after a disconnect RESUMES from resume.pt automatically.
#    Colab usually has 2 CPUs -> use 2 dataloader workers (avoids the freeze warning).
#    Timing: T4 ~1-2 days for 80ep (4 attacks/step); L4 faster (~12-18h). Watch train/val_worst_union.
import os
WANDB_MODE = 'online'   # or 'disabled' if you did not run wandb.login()
NUM_WORKERS = 2         # Colab CPUs; set 4 only on a bigger box
cmd = (f"ATTACKDRO_ROOT={REPO} WANDB_MODE={WANDB_MODE} C5_NUM_WORKERS={NUM_WORKERS} "
       f"python {REPO}/scripts/dev/c5_fromscratch.py --variant M1b --seed 0 --base msd "
       f"--num-workers {NUM_WORKERS} --wandb-mode {WANDB_MODE} --outdir '{DRIVE_OUT}'")
print(cmd)
!{cmd}


## After training / retrieving results
- **Resume:** if Colab disconnects, just re-run the training cell — it loads `resume.pt` from `DRIVE_OUT` and continues from the last epoch.
- **Outputs on Drive** (`DRIVE_OUT`): `ckpt/val_best.pt` (the model to audit), `embed_dump.json` (trajectory figure), `train.json` (history + `best_val_worst_union` + `val_best_sha256`), `ckpt/ep0{20,40,60,80}.pt`.
- **Send back:** download the whole `DRIVE_OUT` folder (or at least `ckpt/val_best.pt` + `train.json` + `embed_dump.json`) → it is audited on the PC under the identical frozen 12-AA harness (config `9162ce44`), then the **paired M1a-vs-M1b (adv-vs-clean)** comparison is computed.
- **Note:** `val_best` is selected by val-worst-union (train-holdout `val_select[49000:50000]`, 20-step) — same selection rule as M1a and B3/B4. The real comparable number is the 12-AA audit done on the PC.
